In [ ]:
file_path =  '/content/drive/MyDrive/colab_datas/small_llm/input.txt'

In [ ]:
# Read the whole file as a single string
with open(file_path, 'r') as file:
    text = file.read()

print(text[:1000])


In [ ]:
# unique chars in the text
chars = sorted(set(text))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
# char to int mapping

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: takes string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a String


print(encode("hii there!"))
print(decode(encode("hii there!")))

[46, 47, 47, 1, 58, 46, 43, 56, 43, 2]
hii there!


In [ ]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)

print(data.shape, data.dtype)
print(data[0:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [ ]:
# split data to train and validate dataset

n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"input={context} taget={target}")

input=tensor([18]) taget=47
input=tensor([18, 47]) taget=56
input=tensor([18, 47, 56]) taget=57
input=tensor([18, 47, 56, 57]) taget=58
input=tensor([18, 47, 56, 57, 58]) taget=1
input=tensor([18, 47, 56, 57, 58,  1]) taget=15
input=tensor([18, 47, 56, 57, 58,  1, 15]) taget=47
input=tensor([18, 47, 56, 57, 58,  1, 15, 47]) taget=58


In [ ]:
torch.manual_seed(1337)
batch_size = 4 # no of independent seq to process in parallel
block_size = 8 # max context len for prediction

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)

print('targets:')
print(yb.shape)
print(yb)

print("-------")

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(f"input={context} taget={target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-------
input=tensor([24]) taget=43
input=tensor([24, 43]) taget=58
input=tensor([24, 43, 58]) taget=5
input=tensor([24, 43, 58,  5]) taget=57
input=tensor([24, 43, 58,  5, 57]) taget=1
input=tensor([24, 43, 58,  5, 57,  1]) taget=46
input=tensor([24, 43, 58,  5, 57,  1, 46]) taget=43
input=tensor([24, 43, 58,  5, 57,  1, 46, 43]) taget=39
input=tensor([44]) taget=53
input=tensor([44, 53]) taget=56
input=tensor([44, 53, 56]) taget=1
input=tensor([44, 53, 56,  1]) taget=58
input=tensor([44, 53, 56,  1, 58]) taget=46
input=tensor([44, 53, 56,  1, 58, 46]) taget=39
input=tensor([44, 53, 56,  1, 58, 46, 

In [ ]:
print(xb) # input to transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.token_embedding_table  = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets = None):
    logits = self.token_embedding_table(idx) # (B,T,C)

    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits,loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss = self(idx)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim = 1)
      idx_next = torch.multinomial(probs, num_samples = 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [ ]:
# pyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for i in range(16):
  for steps in range(1000):

    xb,yb = get_batch('train')

    logits,loss = m(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()

  print(loss.item())

In [ ]:
idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))




Heofl go? be's my bermen we, E: a heawn?


Wes int cowh abldave ancowad l sh.


UClly.
Wh
'de nd, aiut;
Winghic
Lot yod aucacat t y hen d s w?
thouryor, che nin me whaineay f I ath s t'ly minos; bye
LINothis eldin;


Cotitree y m we KEEESes hi'laia at thiforleed, Brveyaced dot.
CEORCLAnng.
TI f'd; fedin thatho in, LAnd,
CONGoous, hat fois ind, fouexe, t d bethaketeerus wisous. l hirler'sthalathe I's t mbe telame cashofr aimolilake, sindin faivendshe fikis h bonst shodladre IORGSTalestheeseag a


In [ ]:
!python3 /content/drive/MyDrive/colab_datas/small_llm/bigram.py

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros(B,T,C)
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1] # (t,c)
    xbow[b,t] = torch.mean(xprev,0)

In [ ]:
#version 2
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1,keepdim=True)
xbow2 = wei @ x # (B,T,T) @ (B,T,C) --------> (B,T,C)
torch.allclose(xbow,xbow2)

False

In [ ]:
#version 3: use softmax
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
xbow2 = wei @ x # (B,T,T) @ (B,T,C) --------> (B,T,C)
torch.allclose(xbow,xbow2)

False

In [ ]:
#version 4: use self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32
x= torch.randn(B,T,C)

# single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias = False)
query = nn.Linear(C, head_size, bias = False)
value = nn.Linear(C, head_size, bias = False)

k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)

wei = q @ k.transpose(-2,-1) # * head_size **-0.5 # (B,T,16) @ (B,16,T) ---> (B, T, T)

tril = torch.tril(torch.ones(T,T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)

v = value(x)
out = wei @ v
#out = wei @ x # (B,T,T) @ (B,T,C) --------> (B,T,C)

out.shape
# torch.allclose(xbow,xbow2)

torch.Size([4, 8, 16])

In [ ]:
wei[0]

tensor([[0.0248, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0052, 0.0091, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0521, 0.0135, 0.2482, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3171, 0.0214, 0.1642, 0.1188, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0412, 0.0487, 0.1046, 0.0742, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1060, 0.5347, 0.2059, 0.1030, 0.7402, 0.0192, 0.0000, 0.0000],
        [0.4298, 0.3409, 0.1769, 0.2027, 0.0480, 0.8472, 0.2329, 0.0000],
        [0.0238, 0.0316, 0.1002, 0.5013, 0.0117, 0.1336, 0.7671, 1.0000]],
       grad_fn=<SelectBackward0>)

In [ ]:
xbow[0],xbow2[0],xbow.shape,xbow2.shape

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 torch.Size([4, 8, 2]),
 torch.Size([4, 8, 2]))

In [ ]:
!python3 /content/drive/MyDrive/colab_datas/small_llm/bigram2.py

In [ ]:
!python3 /content/drive/MyDrive/colab_datas/small_llm/v2.py

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

#hyper params
batch_size = 32 # no of independent seq to process in parallel
block_size = 64 # def 8,  max context len for prediction
max_iters = 25000
eval_interval = 1000
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 32



file_path =  '/content/drive/MyDrive/colab_datas/small_llm/input.txt'

# Read the whole file as a single string
with open(file_path, 'r') as file:
    text = file.read()

print(text[:1000])



# unique chars in the text
chars = sorted(set(text))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


# char to int mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: takes string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a String

print(encode("hii there!"))
print(decode(encode("hii there!")))

#create tensor from text
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[0:100])

# split data to train and validate dataset
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
train_data[:block_size+1]

x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"input={context} taget={target}")


##
torch.manual_seed(1337)

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])

  x,y = x.to(device),y.to(device)

  return x,y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)

print('targets:')
print(yb.shape)
print(yb)

print("-------")

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(f"input={context} taget={target}")

print(xb) # input to transformer

## eval loss
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      x,y = get_batch(split)
      logits, loss = model(x,y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out


##

torch.manual_seed(1337)


class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias = False)
    self.query = nn.Linear(n_embed, head_size, bias = False)
    self.value = nn.Linear(n_embed, head_size, bias = False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size, block_size)))

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2,-1) * C**-0.5
    wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf'))
    wei = F.softmax(wei, dim=-1)

    v = self.value(x)
    out = wei @ v
    return out

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

  def forward(self, x):
    return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedForward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embed, n_embed),
      nn.ReLU(),
    )

  def forward(self, x):
    return self.net(x)

class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table  = nn.Embedding(vocab_size, n_embed)
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    #self.sa_head = Head(n_embed)
    self.sa_heads = MultiHeadAttention(4,n_embed//4)
    self.ffwd = FeedForward(n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, targets = None):
    B,T = idx.shape
    # logits = self.token_embedding_table(idx) # (B,T,C)
    token_emb = self.token_embedding_table(idx) # (B,T,C)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(T,C)
    x = token_emb + pos_emb
    #x = self.sa_head(x)
    x = self.sa_heads(x)
    x = self.ffwd(x)
    logits = self.lm_head(x) # (B,T,C=vocab size)


    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits,loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:,-block_size:]
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim = 1)
      idx_next = torch.multinomial(probs, num_samples = 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

model = BigramLanguageModel()
m = model.to(device)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

# idx = torch.zeros((1,1), dtype=torch.long)
# print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))

# pyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# train
for i in range(max_iters):
  if i % eval_interval ==0:
    losses = estimate_loss()
    print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

  xb,yb = get_batch('train')
  # xb.to(device)
  # yb.to(device)
  logits,loss = model(xb,yb)
  optimizer.zero_grad(set_to_none = True)
  loss.backward()
  optimizer.step()

  #print(loss.item())


#generate
context = torch.zeros((1,1), dtype=torch.long, device = device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))






First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
#generate
context = torch.zeros((1,1), dtype=torch.long, device = device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


He my hat vainress dooupan may pesot usaity and wha with froe,
What gat wornow
Ats age as of tlantio ther's our, on touse; it not herend sinnt:
A in nearbery of
ouard seend more coer my solece, whill'd we it rainess!
Thy heand the that by at you the theave you purilf in theess: thou do det: andizesencerver theng's I me
advedsioure heepich heake doth, wrest bed upoill, ther be wick; in dents of say despath to befe,
IcI wreall it beved;
Worite, sonicesed notheeeest uchour'st, 'd intservaine, roave


In [3]:
def chat(model, max_new_tokens=200):
    model.eval() # Set model to evaluation mode
    print("--- Start Chatting! Type 'quit' or 'exit' to stop ---")

    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ['quit', 'exit']:
            print("Ending chat...")
            break

        if not user_input.strip():
            continue

        # Filter out characters that aren't in the trained vocabulary
        filtered_input = ''.join([c for c in user_input if c in stoi])

        if not filtered_input:
            print("Bot: (Error - input contains no characters from the model's vocabulary)")
            continue

        # Encode user input and convert to tensor on the proper device
        context = torch.tensor([encode(filtered_input)], dtype=torch.long, device=device)

        # Generate completion
        with torch.no_grad():
            # generate returns input_tokens + generated_tokens
            out_tokens = model.generate(context, max_new_tokens=max_new_tokens)[0].tolist()

        # Decode full response
        response = decode(out_tokens)

        # Print the response (you can choose to print full text or just the completion)
        print(f"Bot: {response}")

# Call the function after training your model
#chat(model, max_new_tokens=150)

# With BLOCK

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

#hyper params
batch_size = 32 # no of independent seq to process in parallel
block_size = 64 # def 8,  max context len for prediction
max_iters = 15000
eval_interval = 1000
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 32



file_path =  '/content/drive/MyDrive/colab_datas/small_llm/input.txt'

# Read the whole file as a single string
with open(file_path, 'r') as file:
    text = file.read()

print(text[:1000])



# unique chars in the text
chars = sorted(set(text))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


# char to int mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: takes string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a String

print(encode("hii there!"))
print(decode(encode("hii there!")))

#create tensor from text
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[0:100])

# split data to train and validate dataset
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
train_data[:block_size+1]

x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"input={context} taget={target}")


##
torch.manual_seed(1337)

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])

  x,y = x.to(device),y.to(device)

  return x,y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)

print('targets:')
print(yb.shape)
print(yb)

print("-------")

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(f"input={context} taget={target}")

print(xb) # input to transformer

## eval loss
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      x,y = get_batch(split)
      logits, loss = model(x,y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out


##

torch.manual_seed(1337)


class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias = False)
    self.query = nn.Linear(n_embed, head_size, bias = False)
    self.value = nn.Linear(n_embed, head_size, bias = False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size, block_size)))

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2,-1) * C**-0.5
    wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf'))
    wei = F.softmax(wei, dim=-1)

    v = self.value(x)
    out = wei @ v
    return out

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

  def forward(self, x):
    return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedForward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embed, n_embed),
      nn.ReLU(),
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  def __init__(self, n_embed, n_head):
    super().__init__()
    head_size = n_embed // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embed)

  def forward(self, x):
    x = self.sa(x)
    x = self.ffwd(x)
    return x


class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table  = nn.Embedding(vocab_size, n_embed)
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    #self.sa_head = Head(n_embed)
    # self.sa_heads = MultiHeadAttention(4,n_embed//4)
    # self.ffwd = FeedForward(n_embed)
    self.blocks = nn.Sequential(
        Block(n_embed, n_head = 4),
        Block(n_embed, n_head = 4),
        Block(n_embed, n_head = 4),
    )
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, targets = None):
    B,T = idx.shape
    # logits = self.token_embedding_table(idx) # (B,T,C)
    token_emb = self.token_embedding_table(idx) # (B,T,C)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(T,C)
    x = token_emb + pos_emb
    #x = self.sa_head(x)
    # x = self.sa_heads(x)
    # x = self.ffwd(x)
    x = self.blocks(x)
    logits = self.lm_head(x) # (B,T,C=vocab size)


    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits,loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:,-block_size:]
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim = 1)
      idx_next = torch.multinomial(probs, num_samples = 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

model = BigramLanguageModel()
m = model.to(device)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

# idx = torch.zeros((1,1), dtype=torch.long)
# print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))

# pyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# train
for i in range(max_iters):
  if i % eval_interval ==0:
    losses = estimate_loss()
    print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

  xb,yb = get_batch('train')
  # xb.to(device)
  # yb.to(device)
  logits,loss = model(xb,yb)
  optimizer.zero_grad(set_to_none = True)
  loss.backward()
  optimizer.step()

  #print(loss.item())


#generate
context = torch.zeros((1,1), dtype=torch.long, device = device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))






First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

#hyper params
batch_size = 32 # no of independent seq to process in parallel
block_size = 64 # def 8,  max context len for prediction
max_iters = 15000
eval_interval = 1000
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 32



file_path =  '/content/drive/MyDrive/colab_datas/small_llm/input.txt'

# Read the whole file as a single string
with open(file_path, 'r') as file:
    text = file.read()

print(text[:1000])



# unique chars in the text
chars = sorted(set(text))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


# char to int mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: takes string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a String

print(encode("hii there!"))
print(decode(encode("hii there!")))

#create tensor from text
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[0:100])

# split data to train and validate dataset
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
train_data[:block_size+1]

x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"input={context} taget={target}")


##
torch.manual_seed(1337)

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])

  x,y = x.to(device),y.to(device)

  return x,y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)

print('targets:')
print(yb.shape)
print(yb)

print("-------")

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(f"input={context} taget={target}")

print(xb) # input to transformer

## eval loss
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      x,y = get_batch(split)
      logits, loss = model(x,y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out


##

torch.manual_seed(1337)


class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias = False)
    self.query = nn.Linear(n_embed, head_size, bias = False)
    self.value = nn.Linear(n_embed, head_size, bias = False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size, block_size)))

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2,-1) * C**-0.5
    wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf'))
    wei = F.softmax(wei, dim=-1)

    v = self.value(x)
    out = wei @ v
    return out

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embed, n_embed)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.proj(out)
    return out

class FeedForward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embed, 4 * n_embed),
      nn.ReLU(),
      nn.Linear(4 * n_embed, n_embed),
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  def __init__(self, n_embed, n_head):
    super().__init__()
    head_size = n_embed // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embed)

  def forward(self, x):
    x = x + self.sa(x)
    x = x + self.ffwd(x)
    return x


class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table  = nn.Embedding(vocab_size, n_embed)
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    #self.sa_head = Head(n_embed)
    # self.sa_heads = MultiHeadAttention(4,n_embed//4)
    # self.ffwd = FeedForward(n_embed)
    self.blocks = nn.Sequential(
        Block(n_embed, n_head = 4),
        Block(n_embed, n_head = 4),
        Block(n_embed, n_head = 4),
    )
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, targets = None):
    B,T = idx.shape
    # logits = self.token_embedding_table(idx) # (B,T,C)
    token_emb = self.token_embedding_table(idx) # (B,T,C)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(T,C)
    x = token_emb + pos_emb
    #x = self.sa_head(x)
    # x = self.sa_heads(x)
    # x = self.ffwd(x)
    x = self.blocks(x)
    logits = self.lm_head(x) # (B,T,C=vocab size)


    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits,loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:,-block_size:]
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim = 1)
      idx_next = torch.multinomial(probs, num_samples = 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

model = BigramLanguageModel()
m = model.to(device)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

# idx = torch.zeros((1,1), dtype=torch.long)
# print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))

# pyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# train
for i in range(max_iters):
  if i % eval_interval ==0:
    losses = estimate_loss()
    print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

  xb,yb = get_batch('train')
  # xb.to(device)
  # yb.to(device)
  logits,loss = model(xb,yb)
  optimizer.zero_grad(set_to_none = True)
  loss.backward()
  optimizer.step()

  #print(loss.item())


#generate
context = torch.zeros((1,1), dtype=torch.long, device = device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))






First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
chat(model, max_new_tokens=200)

--- Start Chatting! Type 'quit' or 'exit' to stop ---

You: if
Bot: ifform their'd,
'Tis it braice the ways we torruch chant me dishard, well thee father side stars is being greets!
Serve Cay In thy scon
you:
If in alond-must that
me, selifel bel, come than the in out
C

You: father side
Bot: father side I nighter's saice, be heart but, sbay swor,
Or to fielts. Nop all no.

First Getter book, whereful thou;
My us!

DUKE VINCENTIO:
My sir.

Thas me eles: us Her of hear thee for dham shall you reping t


KeyboardInterrupt: Interrupted by user

In [1]:
# with layer norm

import torch
import torch.nn as nn
from torch.nn import functional as F

#hyper params
batch_size = 32 # no of independent seq to process in parallel
block_size = 64 # def 8,  max context len for prediction
max_iters = 15000
eval_interval = 1000
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 32



file_path =  '/content/drive/MyDrive/colab_datas/small_llm/input.txt'

# Read the whole file as a single string
with open(file_path, 'r') as file:
    text = file.read()

print(text[:1000])



# unique chars in the text
chars = sorted(set(text))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


# char to int mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: takes string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a String

print(encode("hii there!"))
print(decode(encode("hii there!")))

#create tensor from text
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[0:100])

# split data to train and validate dataset
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
train_data[:block_size+1]

x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"input={context} taget={target}")


##
torch.manual_seed(1337)

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])

  x,y = x.to(device),y.to(device)

  return x,y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)

print('targets:')
print(yb.shape)
print(yb)

print("-------")

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(f"input={context} taget={target}")

print(xb) # input to transformer

## eval loss
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      x,y = get_batch(split)
      logits, loss = model(x,y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out


##

torch.manual_seed(1337)


class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias = False)
    self.query = nn.Linear(n_embed, head_size, bias = False)
    self.value = nn.Linear(n_embed, head_size, bias = False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size, block_size)))

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2,-1) * C**-0.5
    wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf'))
    wei = F.softmax(wei, dim=-1)

    v = self.value(x)
    out = wei @ v
    return out

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embed, n_embed)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.proj(out)
    return out

class FeedForward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embed, 4 * n_embed),
      nn.ReLU(),
      nn.Linear(4 * n_embed, n_embed),
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  def __init__(self, n_embed, n_head):
    super().__init__()
    head_size = n_embed // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embed)
    self.ln1 = nn.LayerNorm(n_embed)
    self.ln2 = nn.LayerNorm(n_embed)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x


class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table  = nn.Embedding(vocab_size, n_embed)
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    #self.sa_head = Head(n_embed)
    # self.sa_heads = MultiHeadAttention(4,n_embed//4)
    # self.ffwd = FeedForward(n_embed)
    self.blocks = nn.Sequential(
        Block(n_embed, n_head = 4),
        Block(n_embed, n_head = 4),
        Block(n_embed, n_head = 4),
    )
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, targets = None):
    B,T = idx.shape
    # logits = self.token_embedding_table(idx) # (B,T,C)
    token_emb = self.token_embedding_table(idx) # (B,T,C)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(T,C)
    x = token_emb + pos_emb
    #x = self.sa_head(x)
    # x = self.sa_heads(x)
    # x = self.ffwd(x)
    x = self.blocks(x)
    logits = self.lm_head(x) # (B,T,C=vocab size)


    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits,loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:,-block_size:]
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim = 1)
      idx_next = torch.multinomial(probs, num_samples = 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

model = BigramLanguageModel()
m = model.to(device)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

# idx = torch.zeros((1,1), dtype=torch.long)
# print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))

# pyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# train
for i in range(max_iters):
  if i % eval_interval ==0:
    losses = estimate_loss()
    print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

  xb,yb = get_batch('train')
  # xb.to(device)
  # yb.to(device)
  logits,loss = model(xb,yb)
  optimizer.zero_grad(set_to_none = True)
  loss.backward()
  optimizer.step()

  #print(loss.item())


#generate
context = torch.zeros((1,1), dtype=torch.long, device = device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))






First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chat(model, max_new_tokens=500)

--- Start Chatting! Type 'quit' or 'exit' to stop ---

You: true af 
Bot: true af conle
I: I have thou dress prouny: conce,
Woonce to yessishy lean did toses
Madashs shake are and fatul in you mords,
Whose, in imper or Clacks this you, I'bl man's retther batue Cames----De optless.

DUKE VIRGILA:
All than of hatherce it to shame slusurer
Come!
Spounts
sin monser: shere untle, that neven onestress, conticution's so stroch appert:
The his both thee.

VIRGFOLA:
He restoness, not fil
And Engeares sheer sinks the,
He or starry,
And for Thaness I'll raughter:
If it?

LADY EKEN:
Whos

You: are you sure about what you are talking about 
Bot: are you sure about what you are talking about his standed!
While on shat,
And's you pleace tady yet at papiet?

KING EDWUET:
Unnot man, not thousand:
Srain yrough pome mearres,
Than eave instomers.-
For rightiency I I can her chinecy make pureop thee,
To death.

RY BAHOLYINGBROSILUS:
As solp themsiar: leave out forthan's comans our thee bean tither;
We him!

In [1]:
# with some cosmetic changes

import torch
import torch.nn as nn
from torch.nn import functional as F

#hyper params
batch_size = 64 # no of independent seq to process in parallel
block_size = 256 # def 8,  max context len for prediction
max_iters = 15000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 384
n_head = 6
n_layer = 8
dropout = 0.2



file_path =  '/content/drive/MyDrive/colab_datas/small_llm/input.txt'

# Read the whole file as a single string
with open(file_path, 'r') as file:
    text = file.read()

print(text[:1000])



# unique chars in the text
chars = sorted(set(text))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


# char to int mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: takes string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a String

print(encode("hii there!"))
print(decode(encode("hii there!")))

#create tensor from text
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[0:100])

# split data to train and validate dataset
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


#block_size = 8
train_data[:block_size+1]

x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  #print(f"input={context} taget={target}")


##
torch.manual_seed(1337)

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])

  x,y = x.to(device),y.to(device)

  return x,y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)

print('targets:')
print(yb.shape)
print(yb)

print("-------")

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    # print(f"input={context} taget={target}")

print(xb) # input to transformer

## eval loss
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      x,y = get_batch(split)
      logits, loss = model(x,y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out


##

torch.manual_seed(1337)


class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias = False)
    self.query = nn.Linear(n_embed, head_size, bias = False)
    self.value = nn.Linear(n_embed, head_size, bias = False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)



  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2,-1) * C**-0.5
    wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf'))
    wei = F.softmax(wei, dim=-1)

    wei = self.dropout(wei)

    v = self.value(x)
    out = wei @ v
    return out

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embed, n_embed)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out

class FeedForward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embed, 4 * n_embed),
      nn.ReLU(),
      nn.Linear(4 * n_embed, n_embed),
      nn.Dropout(dropout)
    )

  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  def __init__(self, n_embed, n_head):
    super().__init__()
    head_size = n_embed // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embed)
    self.ln1 = nn.LayerNorm(n_embed)
    self.ln2 = nn.LayerNorm(n_embed)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x


class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table  = nn.Embedding(vocab_size, n_embed)
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    #self.sa_head = Head(n_embed)
    # self.sa_heads = MultiHeadAttention(4,n_embed//4)
    # self.ffwd = FeedForward(n_embed)
    self.blocks = nn.Sequential(
        *[Block(n_embed, n_head = n_head) for _ in range(n_layer) ]
    )
    self.ln_f = nn.LayerNorm(n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, targets = None):
    B,T = idx.shape
    # logits = self.token_embedding_table(idx) # (B,T,C)
    token_emb = self.token_embedding_table(idx) # (B,T,C)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(T,C)
    x = token_emb + pos_emb
    #x = self.sa_head(x)
    # x = self.sa_heads(x)
    # x = self.ffwd(x)
    x = self.blocks(x)
    logits = self.lm_head(x) # (B,T,C=vocab size)


    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits,loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:,-block_size:]
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim = 1)
      idx_next = torch.multinomial(probs, num_samples = 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

model = BigramLanguageModel()
m = model.to(device)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

# idx = torch.zeros((1,1), dtype=torch.long)
# print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))

# pyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# train
for i in range(max_iters):
  if i % eval_interval ==0:
    losses = estimate_loss()
    print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

  xb,yb = get_batch('train')
  # xb.to(device)
  # yb.to(device)
  logits,loss = model(xb,yb)
  optimizer.zero_grad(set_to_none = True)
  loss.backward()
  optimizer.step()

  #print(loss.item())


#generate
context = torch.zeros((1,1), dtype=torch.long, device = device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))


Streaming output truncated to the last 5000 lines.
        60, 43,  1, 52, 53, 58,  1, 61, 39, 57, 58, 43, 42,  1, 47, 58,  6,  1,
        44, 53, 56,  1, 61, 39, 56, 56,  5, 42,  1, 46, 43,  1, 46, 39, 58, 46,
         1, 52, 53, 58,  6,  0, 14, 59, 58,  1, 40, 39, 57, 43, 50, 63,  1, 63,
        47, 43, 50, 42, 43, 42,  1, 59, 54, 53, 52,  1, 41, 53, 51, 54, 56, 53,
        51, 47, 57, 43,  0, 32, 46, 39, 58,  1, 61, 46, 47, 41, 46,  1, 46, 47,
        57,  1, 52, 53, 40, 50, 43,  1, 39, 52, 41, 43, 57, 58, 53, 56, 57,  1,
        39, 41, 46, 47, 43, 60, 43, 42,  1, 61, 47, 58, 46,  1, 40, 50, 53, 61,
        57, 10,  0, 25, 53], device='cuda:0') taget=56
input=tensor([ 5, 57,  1, 52, 39, 51, 43,  6,  1, 42, 53, 58, 46,  1, 40, 43, 41, 53,
        51, 43,  1, 53, 44,  1, 58, 46, 47, 57, 12,  0,  0, 26, 27, 30, 32, 20,
        33, 25, 14, 17, 30, 24, 13, 26, 16, 10,  0, 35, 39, 56, 57,  1, 46, 39,
        60, 43,  1, 52, 53, 58,  1, 61, 39, 57, 58, 43, 42,  1, 47, 58,  6,  1,
        

KeyboardInterrupt: 

In [4]:
chat(model, max_new_tokens=500)

--- Start Chatting! Type 'quit' or 'exit' to stop ---

You: hi
Bot: his; then I'll budge you build as little.
Prithee, father, and when thou cavest me not?
The king revell'd of thy face will with slaughter'd hand,
With tears low thy windown and my hearts.
Are cull'd the lady Tower. Who's my heart?

JULIET:
It may swear be thus the while: saw you think
Her matter, that I am out of the door.

SICINIUS:
If the veries did to visit you,
See of the occordination and twelve as my right.

BRUTUS:
It think it hath done some perform'd.

SICINIUS:
Your person must be consul,

You: hi 
Bot: hi minister's palace with the teeth.
Go, side me from heaven, they have done the world;
And therefore, be the third to the rest,
Where she, by the hope of York
May have been devoted this day's word:
And, God here he spake the tenth to France,
And throw his subject shall rain horse thus?
Hear not he not waken his help till a life;
His head, and whose perjury had left me,
I have ever so power to the father. But c